## Vamos a crear un codigo para aprender a usar los assistants de gpt va a ser el Asistente HW.

In [1]:
import os
import pandas as pd
import json
import requests
from openai import OpenAI
import time

CONFIG_PATH = os.path.join("..","..","config.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\config.json'

In [ ]:


client = OpenAI(api_key="TU_API_KEY")

# 1. Subir el archivo (puede ser .txt, .csv, .json, .docx convertido a binario)
my_file = client.files.create(
    file=open("tu_archivo.json", "rb"),  # reemplaza por .csv o .docx según corresponda
    purpose="assistants"
)

# 2. Crear el assistant (puedes ajustar instrucciones)
my_assistant = client.beta.assistants.create(
    model="gpt-4o",  # o el que quieras usar
    name="MiAssistantConArchivo",
    instructions="Usa el archivo provisto como contexto para responder preguntas.",
    tools=[{"type": "retrieval"}]  # habilita que pueda usar archivos adjuntos
)

# 3. Crear un thread
my_thread = client.beta.threads.create()

# 4. Agregar el mensaje del usuario incluyendo el file_id
user_msg = client.beta.threads.messages.create(
    thread_id=my_thread.id,
    role="user",
    content="¿Cuál es la información clave de este archivo?",  # tu pregunta simple
    file_ids=[my_file.id]
)

# 5. Ejecutar el assistant (run)
my_run = client.beta.threads.runs.create(
    thread_id=my_thread.id,
    assistant_id=my_assistant.id,
)

# 6. Esperar a que se complete (polling básico)
for _ in range(30):
    run_status = client.beta.threads.runs.retrieve(
        thread_id=my_thread.id,
        run_id=my_run.id
    )
    if run_status.status == "completed":
        break
    time.sleep(1)

# 7. Obtener la respuesta del assistant
all_messages = client.beta.threads.messages.list(thread_id=my_thread.id)
# Asume que la primera respuesta del assistant está en all_messages.data[0]
assistant_response = None
for msg in all_messages.data:
    if msg.role == "assistant":
        assistant_response = msg.content[0].text.value
        break

print("Respuesta del assistant:", assistant_response)